In [ ]:
from dotenv import load_dotenv, find_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from typing import Any, Callable
from langchain.agents.middleware import (
    before_model,
    wrap_model_call,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from langgraph.runtime import Runtime

C:\Users\User\AppData\Roaming\Python\Python314\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
env_path = find_dotenv()
load_dotenv(env_path, override=True)

True

Prebuilt Middleware

In [3]:
@tool
def get_current_year() -> str:
    """Return the current year."""
    return "2026"

In [4]:
summary_prompt = """
Summarize the main thrust of this conversation. What have the human and assistant
discussed so far? Focus on key facts and requests.
<messages>
Messages to summarize:
{messages}
</messages>
"""

In [5]:
agent = create_agent(
    model="openai:gpt-3.5-turbo",
    tools=[get_current_year],
    middleware=[
        SummarizationMiddleware(
            model="openai:gpt-4o-mini",
            summary_prompt=summary_prompt,
            trigger=("tokens", 50),
            keep=("tokens", 10),
            trim_tokens_to_summarize=None,
        ),
    ]
)

In [6]:
agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What year is it?"
            }
        ]
    }
)

{'messages': [HumanMessage(content='Here is a summary of the conversation to date:\n\nThe conversation consists of a single message from the human asking for the current year. There are no further discussions or requests from either party.', additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='dce26644-3a50-451f-95ea-6a9eb9f03b20'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_uFXGZncFYjMlMkBEP0mPfgOf', 'function': {'arguments': '{}', 'name': 'get_current_year'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 43, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DsjIFbzKnuYuCPxlTmZmDM6J6Qzt1', 'service_tier': 'default

 Custom Middleware

In [7]:
@before_model
def log_before_model(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    print(f"[LOG] About to call model with {len(state['messages'])} messages")
    return None

In [8]:
@wrap_model_call
def retry_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    for attempt in range(3):
        try:
            return handler(request)
        except Exception as e:
            print(f"[WARN] Attempt {attempt + 1} failed: {e}")
            if attempt == 2:
                raise

In [11]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"The weather in {city} is sunny."

In [12]:
agent = create_agent(
    model="gpt-5.4",
    middleware=[log_before_model, retry_model],
    tools=[get_weather],
)

In [14]:
response = agent.invoke(
        {
            "messages": [
                {"role": "user", "content": "What is the weather in Tokyo?"}
            ]
        }
    )

response

[LOG] About to call model with 1 messages
[LOG] About to call model with 3 messages


{'messages': [HumanMessage(content='What is the weather in Tokyo?', additional_kwargs={}, response_metadata={}, id='e6d9a5c8-6263-4c5f-842d-dd68a8a01ce7'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_i1tMVm68GLHtStcsxg4JJrAk', 'function': {'arguments': '{"city":"Tokyo"}', 'name': 'get_weather'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 133, 'total_tokens': 150, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-5.4-2026-03-05', 'system_fingerprint': None, 'id': 'chatcmpl-DsjJlHEtA8Jqol1wJEHsDIfAzFvpu', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ee3ad-f747-7631-a28b-f5114724bdf6-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'call_i1tMVm68GL